In [3]:
import pandas as pd
import duckdb


In [112]:
conn = duckdb.connect("taxinyc.db")

In [7]:
conn.execute("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'main'
""").df()

,table_name
0,trips
1,trips_flagged
2,clean_db
3,clean_yellow


In [8]:
conn.execute(
    """ 
    CREATE TABLE IF NOT EXISTS combined_trips AS
    SELECT * 
    FROM read_parquet('../data/raw/yellow/*.parquet')
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Count
0,48722602


In [11]:
conn.execute(
    """ 
    SELECT * FROM combined_trips LIMIT 5
    """
).df()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1,1.60,1,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1,0.50,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1,0.60,1,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [14]:
conn.execute(
    """ 
    SUMMARIZE combined_trips;
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,7,4,1.8601995845788368,0.6782855211602725,2,2,2,48722602,0.00
1,tpep_pickup_datetime,TIMESTAMP,2007-12-05 18:45:00,2025-12-31 23:59:59,19363052,2025-07-06 05:22:36.621952,None,2025-04-08 14:06:48.041509,2025-07-03 17:27:40.797311,2025-10-06 09:00:49.281803,48722602,0.00
2,tpep_dropoff_datetime,TIMESTAMP,2007-12-05 19:02:00,2026-01-05 13:00:34,20705652,2025-07-06 05:39:57.973512,None,2025-04-08 10:50:31.31837,2025-07-03 22:59:47.470839,2025-10-06 08:32:02.500046,48722602,0.00
3,passenger_count,BIGINT,0,9,11,1.2954911827605122,0.7291638933484583,1,1,1,48722602,23.83
4,trip_distance,DOUBLE,0.0,397994.37,8495,6.842892016522826,649.3613312608348,1.0382721236737265,1.8467423679199952,3.6973671588322063,48722602,0.00
5,RatecodeID,BIGINT,1,99,7,3.225041004337616,14.265148724130462,1,1,1,48722602,23.83
6,store_and_fwd_flag,VARCHAR,N,Y,2,None,None,None,None,None,48722602,23.83
7,PULocationID,INTEGER,1,265,290,161.5041679834751,66.19732966352633,118,161,232,48722602,0.00
8,DOLocationID,INTEGER,1,265,298,161.10941121330097,70.45459645479511,109,162,233,48722602,0.00
9,payment_type,BIGINT,0,5,6,0.9372241244422865,0.7465483770308788,1,1,1,48722602,0.00


In [51]:
conn.execute(
    """ 
    SELECT COUNT (*) FROM (
            SELECT DISTINCT (*)
            FROM combined_trips
        )
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,count_star()
0,48722601


In [46]:
conn.execute(
    """ 
    SELECT *,
        COUNT (*) AS duplicate_count,
    FROM combined_trips
    GROUP BY ALL
    HAVING COUNT(*) > 1
    ORDER BY duplicate_count DESC
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,duplicate_count
0,2,2025-07-05 17:19:29,2025-07-05 17:19:34,1,0.0,1,N,132,132,2,...,0.0,0.5,0.0,0.0,1.0,6.25,0.0,1.75,0.0,2


In [43]:
conn.execute(
    """ 
    SELECT 
        COUNT (*) AS total_trips,
        COUNT (CASE WHEN tpep_pickup_datetime > tpep_dropoff_datetime THEN 1 END) AS impossible_duration,
        COUNT (CASE WHEN tpep_pickup_datetime = tpep_dropoff_datetime THEN 1 END) AS zero_duration,
        COUNT (CASE WHEN tpep_pickup_datetime < tpep_dropoff_datetime THEN 1 END) AS positive_duration,
        impossible_duration + zero_duration + positive_duration AS total_count
    FROM combined_trips
    """
).df()

,total_trips,impossible_duration,zero_duration,positive_duration,total_count
0,48722602,2235,544069,48176298,48722602


In [42]:
conn.execute(
    """ 
    SUMMARIZE (
        SELECT (*)
        FROM combined_trips
        WHERE passenger_count = 0 AND fare_amount = 0 AND total_amount > 0
    )
    """
).df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,2,2,1.032258064516129,0.1774014725270341,1,1,1,124,0.0
1,tpep_pickup_datetime,TIMESTAMP,2025-01-01 05:22:35,2025-12-24 14:35:45,127,2025-06-06 19:13:04.008065,None,2025-02-26 14:14:48.5,2025-05-23 07:16:33.5,2025-08-27 14:18:24.5,124,0.0
2,tpep_dropoff_datetime,TIMESTAMP,2025-01-01 05:23:05,2025-12-24 14:37:28,136,2025-06-06 19:19:50.177419,None,2025-02-26 14:15:08,2025-05-23 07:17:50.5,2025-08-27 14:19:02,124,0.0
3,passenger_count,BIGINT,0,0,1,0.0,0.0,0,0,0,124,0.0
4,trip_distance,DOUBLE,0.0,49.8,41,3.8045161290322578,8.07034472569838,0.0,0.13,3.45,124,0.0
5,RatecodeID,BIGINT,1,99,5,7.048387096774194,14.56404685491046,5,5,5,124,0.0
6,store_and_fwd_flag,VARCHAR,N,Y,2,None,None,None,None,None,124,0.0
7,PULocationID,INTEGER,1,265,51,149.70967741935485,69.00663031577018,127,142,193,124,0.0
8,DOLocationID,INTEGER,1,265,43,172.80645161290323,77.15423023134335,132,163,254,124,0.0
9,payment_type,BIGINT,1,4,4,2.8225806451612905,0.9546496151742933,2,3,4,124,0.0


In [41]:
conn.execute(
    """ 
    SUMMARIZE (
        SELECT (*)
        FROM combined_trips
        WHERE passenger_count = 0 AND fare_amount = 0 AND total_amount = 0
    )
    """
).df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,1,1,1.0,0.0,1,1,1,197,0.0
1,tpep_pickup_datetime,TIMESTAMP,2025-01-04 08:37:48,2025-12-30 11:22:27,185,2025-07-14 20:48:06.771574,None,2025-04-07 00:05:46.9375,2025-07-24 23:58:19,2025-10-14 04:02:40,197,0.0
2,tpep_dropoff_datetime,TIMESTAMP,2025-01-04 08:38:18,2025-12-30 11:22:27,221,2025-07-14 20:44:52.182741,None,2025-04-07 00:05:54.6875,2025-07-25 00:11:25.333333,2025-10-14 02:30:34.4375,197,0.0
3,passenger_count,BIGINT,0,0,1,0.0,0.0,0,0,0,197,0.0
4,trip_distance,DOUBLE,0.0,18.3,27,0.8426395939086294,2.726683927757985,0.0,0.0,0.018750000000000003,197,0.0
5,RatecodeID,BIGINT,1,99,5,27.781725888324875,42.785989398900746,1,2,99,197,0.0
6,store_and_fwd_flag,VARCHAR,N,Y,2,None,None,None,None,None,197,0.0
7,PULocationID,INTEGER,4,264,64,154.26903553299493,67.05688801448741,111,145,229,197,0.0
8,DOLocationID,INTEGER,36,264,43,196.71573604060913,72.7934624089347,138,235,264,197,0.0
9,payment_type,BIGINT,1,5,5,3.0964467005076144,0.8897589466333328,3,3,4,197,0.0


In [ ]:
conn.execute(
    """ 
    SELECT 
        COUNT (*) AS total_count,
        COUNT (CASE WHEN passenger_count > 0 AND fare_amount > 0 AND total_amount > 0 THEN 1 END) AS positive_passenger_positive_fare_count,
        COUNT (CASE WHEN passenger_count = 0 AND fare_amount = 0 AND total_amount = 0 THEN 1 END) AS zero_passenger_zero_fare_count,
        COUNT (CASE WHEN passenger_count = 0 AND fare_amount = 0 AND total_amount > 0 THEN 1 END) AS zero_passenger_positive_fare_count,
    FROM combined_trips
    """
).df()

,total_count,positive_passenger_positive_fare_count,zero_passenger_zero_fare_count,zero_passenger_positive_fare_count
0,48722602,36031345,197,124


In [35]:
conn.execute(
    """ 
    SELECT
        COUNT (*) AS total_trips,
        COUNT (CASE WHEN total_amount > 0 AND fare_amount > 0 THEN 1 END) AS valid_trips,
        COUNT (CASE WHEN total_amount = 0 AND fare_amount = 0 THEN 1 END) AS zero_amount_trips,
        COUNT (CASE WHEN total_amount > 0 AND fare_amount = 0 THEN 1 END) AS zero_fare_trips,
        COUNT (CASE WHEN total_amount < 0 AND fare_amount < 0 THEN 1 END) AS negative_amount_trips,
        COUNT (CASE WHEN total_amount = 0 AND fare_amount > 0 THEN 1 END) AS data_corrupt,
        COUNT (CASE WHEN total_amount < 0 AND fare_amount > 0 THEN 1 END) AS severe_error,
        COUNT (CASE WHEN total_amount > 0 AND fare_amount < 0 THEN 1 END) AS invalid_state,
        COUNT (CASE WHEN total_amount < 0 AND fare_amount = 0 THEN 1 END) AS invalid_trans,
        valid_trips + zero_amount_trips + zero_fare_trips + negative_amount_trips + data_corrupt + severe_error + invalid_state + invalid_trans AS total_class
    FROM combined_trips
    """
).df()

,total_trips,valid_trips,zero_amount_trips,zero_fare_trips,negative_amount_trips,data_corrupt,severe_error,invalid_state,invalid_trans,total_class
0,48722602,45851901,6334,10306,968747,467,0,1879873,4974,48722602


In [32]:
conn.execute(
    """ 
    SUMMARIZE (
        SELECT (*) 
        FROM combined_trips
        WHERE total_amount = 0 AND fare_amount <= 0
        ORDER BY tpep_pickup_datetime ASC
    )
    """
).df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,6,3,1.121408272813388,0.4660657471069113,1,1,1,6334,0.0
1,tpep_pickup_datetime,TIMESTAMP,2025-01-01 01:30:13,2025-12-31 20:58:45,8558,2025-07-02 13:31:09.49258,None,2025-04-04 20:09:55.186157,2025-06-30 18:08:09.635827,2025-09-29 03:31:02.344874,6334,0.0
2,tpep_dropoff_datetime,TIMESTAMP,2025-01-01 01:30:31,2025-12-31 20:58:45,5818,2025-07-02 13:40:39.355226,None,2025-04-04 20:19:53.835426,2025-06-30 18:17:14.230471,2025-09-29 03:42:47.273104,6334,0.0
3,passenger_count,BIGINT,0,6,7,1.0583709650443218,0.4455325496453634,1,1,1,6334,5.6
4,trip_distance,DOUBLE,0.0,19447.8,463,4.60752131354591,244.39368279963915,0.0,0.09953598484848485,1.2214477508650516,6334,0.0
5,RatecodeID,BIGINT,1,99,6,6.328650275965881,21.512976633525533,1,1,1,6334,5.6
6,store_and_fwd_flag,VARCHAR,N,Y,2,None,None,None,None,None,6334,5.6
7,PULocationID,INTEGER,1,265,231,156.52794442690242,68.84372276535717,107,153,230,6334,0.0
8,DOLocationID,INTEGER,1,265,213,161.22166087780235,73.18277410300153,108,161,234,6334,0.0
9,payment_type,BIGINT,0,5,6,3.0050520997789705,1.148931655756106,2,3,4,6334,0.0


In [31]:
conn.execute(
    """ 
    SUMMARIZE (
        SELECT (*) 
        FROM combined_trips
        WHERE total_amount <= 0
        ORDER BY tpep_pickup_datetime ASC
    )
    """
).df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,6,3,1.9961530694874772,0.11732702197943566,2,2,2,980522,0.00
1,tpep_pickup_datetime,TIMESTAMP,2025-01-01 00:00:00,2025-12-31 23:57:56,975761,2025-07-13 13:54:50.883455,None,2025-04-23 18:29:13.794692,2025-07-14 13:14:38.569641,2025-10-15 09:46:30.501965,980522,0.00
2,tpep_dropoff_datetime,TIMESTAMP,2025-01-01 00:00:00,2026-01-01 00:27:38,973500,2025-07-13 14:11:51.11551,None,2025-04-23 18:32:30.123184,2025-07-14 18:04:53.563966,2025-10-15 10:16:42.093086,980522,0.00
3,passenger_count,BIGINT,0,9,9,1.389378472384619,0.776578306909486,1,1,2,980522,17.14
4,trip_distance,DOUBLE,0.0,321242.41,4084,7.028401535101006,656.1754432248994,0.7052405955208721,1.6760534136162462,5.458656325433678,980522,0.00
5,RatecodeID,BIGINT,1,99,7,1.307930514152929,2.5725761348587786,1,1,1,980522,17.14
6,store_and_fwd_flag,VARCHAR,N,Y,2,None,None,None,None,None,980522,17.14
7,PULocationID,INTEGER,1,265,266,155.10472483024347,62.81977095777139,127,145,226,980522,0.00
8,DOLocationID,INTEGER,1,265,290,154.33987100748377,71.57428215924061,94,160,230,980522,0.00
9,payment_type,BIGINT,0,5,6,2.8142336428963346,1.4956527359807401,2,4,4,980522,0.00


In [30]:
conn.execute(
    """ 
    SUMMARIZE (
        SELECT (*) 
        FROM combined_trips
        WHERE fare_amount <= 0
        ORDER BY tpep_pickup_datetime ASC
    )
    """
).df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,6,3,1.9999324096920321,0.10978467778016307,2,2,2,2870234,0.00
1,tpep_pickup_datetime,TIMESTAMP,2025-01-01 00:00:00,2025-12-31 23:57:56,2515509,2025-07-11 16:45:46.226855,None,2025-04-30 18:50:17.680497,2025-07-15 04:33:58.283571,2025-10-08 09:33:39.401851,2870234,0.00
2,tpep_dropoff_datetime,TIMESTAMP,2025-01-01 00:00:00,2026-01-01 00:27:38,3174336,2025-07-11 17:03:28.164007,None,2025-05-01 22:42:41.784507,2025-07-15 19:18:24.024454,2025-10-07 22:28:49.024666,2870234,0.00
3,passenger_count,BIGINT,0,9,9,1.3880828560556537,0.7760028872040522,1,1,2,2870234,71.43
4,trip_distance,DOUBLE,0.0,321242.41,4380,14.232981725531257,1145.151388273739,1.0526202677981802,2.2414246886949276,4.667854502695316,2870234,0.00
5,RatecodeID,BIGINT,1,99,7,1.4254349162365476,4.191314136902795,1,1,1,2870234,71.43
6,store_and_fwd_flag,VARCHAR,N,Y,2,None,None,None,None,None,2870234,71.43
7,PULocationID,INTEGER,1,265,266,153.4966211814089,70.39048863841082,98,149,230,2870234,0.00
8,DOLocationID,INTEGER,1,265,290,152.96422103563683,72.57675545018373,89,158,230,2870234,0.00
9,payment_type,BIGINT,0,5,6,0.9671169667699567,1.5962640110527426,0,0,2,2870234,0.00


In [25]:
conn.execute(
    """ 
    SELECT (*) 
    FROM combined_trips
    WHERE tpep_pickup_datetime <= '2025-01-01'
    ORDER BY tpep_pickup_datetime ASC
    LIMIT 10
    """
).df()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2007-12-05 18:45:00,2007-12-05 19:02:00,1,3.00,1,N,142,234,2,17.0,1.0,0.5,0.00,0.00,1.0,22.75,2.5,0.00,0.75
1,2,2008-12-31 23:04:21,2008-12-31 23:32:25,1,18.12,2,N,132,238,2,70.0,0.0,0.5,0.00,6.94,1.0,80.94,2.5,0.00,0.00
2,2,2009-01-01 00:05:50,2009-01-01 00:10:56,1,1.26,1,N,239,50,2,8.6,1.0,0.5,0.00,0.00,1.0,14.35,2.5,0.00,0.75
3,2,2009-01-01 00:19:34,2009-01-01 01:10:21,6,10.77,1,N,138,239,2,52.7,5.0,0.5,0.00,6.94,1.0,70.39,2.5,1.75,0.00
4,2,2009-01-01 00:20:39,2009-01-01 00:20:49,5,4.47,1,N,230,261,1,24.0,1.0,0.5,2.98,0.00,1.0,32.73,2.5,0.00,0.75
5,2,2009-01-01 08:52:26,2009-01-01 10:00:26,1,11.95,1,N,138,163,1,64.6,0.0,0.5,16.65,13.88,1.0,99.88,2.5,0.00,0.75
6,2,2009-01-01 12:52:15,2009-01-01 13:12:15,1,2.13,1,N,43,230,1,18.4,0.0,0.5,4.63,0.00,1.0,27.78,2.5,0.00,0.75
7,2,2009-01-01 14:08:04,2009-01-01 14:45:04,1,11.56,1,N,138,142,2,50.6,0.0,0.5,0.00,13.88,1.0,68.48,2.5,0.00,0.00
8,2,2024-12-31 20:47:55,2024-12-31 20:54:00,2,1.72,1,N,48,246,1,9.3,1.0,0.5,2.86,0.00,1.0,17.16,2.5,0.00,0.00
9,2,2024-12-31 20:52:50,2024-12-31 21:09:34,2,5.05,1,N,249,262,1,23.3,1.0,0.5,4.00,0.00,1.0,32.30,2.5,0.00,0.00


In [22]:
conn.execute(
    """ 
    SELECT (*) 
    FROM combined_trips
    WHERE tpep_pickup_datetime <= '2023-01-01 00:00'
    """
).df()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2009-01-01 00:19:34,2009-01-01 01:10:21,6,10.77,1,N,138,239,2,52.7,5.0,0.5,0.00,6.94,1.0,70.39,2.5,1.75,0.00
1,2,2007-12-05 18:45:00,2007-12-05 19:02:00,1,3.00,1,N,142,234,2,17.0,1.0,0.5,0.00,0.00,1.0,22.75,2.5,0.00,0.75
2,2,2009-01-01 00:20:39,2009-01-01 00:20:49,5,4.47,1,N,230,261,1,24.0,1.0,0.5,2.98,0.00,1.0,32.73,2.5,0.00,0.75
3,2,2009-01-01 08:52:26,2009-01-01 10:00:26,1,11.95,1,N,138,163,1,64.6,0.0,0.5,16.65,13.88,1.0,99.88,2.5,0.00,0.75
4,2,2009-01-01 12:52:15,2009-01-01 13:12:15,1,2.13,1,N,43,230,1,18.4,0.0,0.5,4.63,0.00,1.0,27.78,2.5,0.00,0.75
5,2,2008-12-31 23:04:21,2008-12-31 23:32:25,1,18.12,2,N,132,238,2,70.0,0.0,0.5,0.00,6.94,1.0,80.94,2.5,0.00,0.00
6,2,2009-01-01 00:05:50,2009-01-01 00:10:56,1,1.26,1,N,239,50,2,8.6,1.0,0.5,0.00,0.00,1.0,14.35,2.5,0.00,0.75
7,2,2009-01-01 14:08:04,2009-01-01 14:45:04,1,11.56,1,N,138,142,2,50.6,0.0,0.5,0.00,13.88,1.0,68.48,2.5,0.00,0.00


In [103]:
conn.execute(
    """     
    CREATE OR REPLACE TABLE flag_combined AS 
    WITH nunique AS (
        SELECT DISTINCT * 
        FROM combined_trips
    ), calculate AS (
        SELECT *,
        trip_distance >= 50 AS extreme_dist_flag,
        EPOCH(tpep_dropoff_datetime - tpep_pickup_datetime)/ 3600 AS duration_hours,
        EPOCH(tpep_dropoff_datetime - tpep_pickup_datetime)/ 60 AS duration_mins,
        FROM nunique
    ), 
    speed AS (
        SELECT *,
            trip_distance / duration_hours AS speed_mph,
            EXTRACT('hour' FROM tpep_pickup_datetime) AS pickup_hour,
            EXTRACT('dow' FROM tpep_pickup_datetime) AS pickup_dow,
            dayname(tpep_pickup_datetime) AS pickup_dayofw,
            CASE WHEN fare_amount > 0 THEN tip_amount / fare_amount ELSE NULL END AS tip_rate,
            CASE WHEN trip_distance > 0 THEN fare_amount / trip_distance ELSE NULL END AS fare_permile,

        FROM calculate
    )
    SELECT *,
        speed_mph > 80 OR speed_mph < 0 OR speed_mph  IS NULL AS impossible_speed,
    FROM speed;
    """
)

conn.execute(
    """ 
    SELECT 
        SUM(impossible_speed) AS count_speed,
        SUM(extreme_dist_flag) AS count_extreme_dist
    FROM flag_combined;
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,count_speed,count_extreme_dist
0,559992.0,9159.0


In [ ]:
conn.execute(
    """ 
    COPY (
        SELECT 
            tpep_pickup_datetime,
            tpep_dropoff_datetime,
            passenger_count,
            trip_distance,
            PULocationID,
            DOLocationID,
            payment_type,
            fare_amount,
            tip_amount,
            total_amount,
            duration_hours,
            duration_mins,
            speed_mph,
            pickup_hour,
            pickup_dow,
            pickup_dayofw,
            tip_rate,
            fare_permile
        FROM flag_combined
        WHERE 
            tpep_pickup_datetime < tpep_dropoff_datetime
            AND tpep_pickup_datetime > '2023-01-01 00:00'
            AND trip_distance BETWEEN 0.01 AND 100
            AND (fare_permile <= 100 AND fare_permile >= 0)
            AND fare_permile BETWEEN 0 AND 200
            AND passenger_count > 0
            AND (total_amount > 0 AND fare_amount >= 0)
            AND (speed_mph < 80 AND speed_mph > 0)
    ) TO "../data/processed/clean_taxinyc_2025.parquet"
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [111]:
conn.close()

In [4]:
diss = duckdb.connect('clean_taxinyc.db')

In [115]:
diss.execute (
    """ 
    CREATE OR REPLACE TABLE clean_taxinyc AS SELECT * FROM '../data/processed/clean_taxinyc_2025.parquet';
    """
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [116]:
diss.execute(
    """ 
    SUMMARIZE clean_taxinyc
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,2,2,1.7818989276154056,0.412956413543637,2,2,2,34962923,0.00
1,tpep_pickup_datetime,TIMESTAMP,2024-12-31 20:47:55,2025-12-31 23:59:54,17356281,2025-07-02 22:40:59.729215,None,2025-04-03 04:39:39.083647,2025-06-29 06:55:46.610344,2025-10-05 04:01:14.087082,34962923,0.00
2,tpep_dropoff_datetime,TIMESTAMP,2024-12-31 20:54:00,2026-01-01 23:00:06,18977580,2025-07-02 22:58:31.286059,None,2025-04-03 03:14:32.071719,2025-06-29 01:17:40.168612,2025-10-05 01:31:29.882198,34962923,0.00
3,passenger_count,BIGINT,1,9,10,1.300851276078948,0.7200164253201979,1,1,1,34962923,0.00
4,trip_distance,DOUBLE,0.01,99.82,5634,3.4338338373481196,4.536590896522592,1.0190583655367407,1.7199288580609182,3.340898771695036,34962923,0.00
5,RatecodeID,BIGINT,1,99,7,3.276224130345166,14.517499808107079,1,1,1,34962923,0.00
6,store_and_fwd_flag,VARCHAR,N,Y,2,None,None,None,None,None,34962923,0.00
7,PULocationID,INTEGER,1,265,290,165.09588520387726,63.20666505272338,132,162,233,34962923,0.00
8,DOLocationID,INTEGER,1,265,298,164.69594893424673,69.23407593369913,115,162,234,34962923,0.00
9,payment_type,BIGINT,1,5,5,1.1738319762337948,0.49042704042688884,1,1,1,34962923,0.00


In [97]:
diss.execute("""
    SELECT 
        MAX(fare_amount) AS max_fare,
        MAX(tip_amount) AS max_tip,
        MAX(tolls_amount) AS max_tolls,
        MAX(total_amount) AS max_total,
        MAX(duration_hours) AS max_duration,
        MAX(speed_mph) AS max_speed,
        MAX(fare_permile) AS max_fare_permile
    FROM clean_taxinyc;
""").df()

,max_fare,max_tip,max_tolls,max_total,max_duration,max_speed,max_fare_permile
0,963.0,960.94,916.87,5297.87,146.653333,80.0,100.0


In [84]:
diss.execute("""
    SELECT * 
    FROM clean_taxinyc 
    WHERE fare_amount > 1000 OR trip_distance > 1000;
""").df()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extreme_dist_flag,duration_hours,duration_mins,speed_mph,pickup_hour,pickup_dow,pickup_dayofw,tip_rate,fare_permile,impossible_speed


In [127]:
#Hourly mart demand
diss.execute (
    """ 
    COPY (SELECT
        pickup_hour,
        SUM(passenger_count) AS total_passengers,
        COUNT(*) AS trip_count,
        AVG(trip_distance) AS avg_trip_distance,
        SUM(total_amount) AS revenue,
        AVG(tip_rate) * 100 AS avg_tip_rate,
        AVG(duration_mins) AS avg_duration,
    FROM clean_taxinyc
    GROUP BY pickup_hour
    ORDER BY pickup_hour ASC)
    
    TO "../data/marts/mart_duck_hourly.parquet"
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """
)

In [131]:
#Daily mart demand
diss.execute (
    """ 
    COPY (
        SELECT
            pickup_dayofw,
            pickup_dow,
            SUM(passenger_count) AS total_passengers,
            COUNT(*) AS trip_count,
            AVG(trip_distance) AS avg_trip_distance,
            SUM(total_amount) AS revenue,
            AVG(tip_rate) * 100 AS avg_tip_rate,
            AVG(duration_mins) AS avg_duration,
        FROM clean_taxinyc
        GROUP BY pickup_dayofw, pickup_dow
        ORDER BY pickup_dow ASC
    ) TO "../data/marts/mart_duck_daily.parquet"
    """
)

In [135]:
#Pickup zone demand
diss.execute (
    """ 
    COPY (
        SELECT
                PULocationID,
                SUM(passenger_count) AS total_passengers,
                COUNT(*) AS trip_count,
                AVG(trip_distance) AS avg_trip_distance,
                SUM(total_amount) AS revenue,
                AVG(tip_rate) * 100 AS avg_tip_rate,
                AVG(duration_mins) AS avg_duration,
            FROM clean_taxinyc
            GROUP BY PULocationID
            ORDER BY trip_count DESC
            LIMIT 20
    ) TO "../data/marts/mart_duck_pickup.parquet"
    """
)

In [136]:
#Common Dropoff
diss.execute (
    """ 
    COPY (
        SELECT
            DOLocationID,
            SUM(passenger_count) AS total_passengers,
            COUNT(*) AS trip_count,
            AVG(trip_distance) AS avg_trip_distance,
            SUM(total_amount) AS revenue,
            AVG(tip_rate) * 100 AS avg_tip_rate,
            AVG(duration_mins) AS avg_duration,
        FROM clean_taxinyc
        GROUP BY DOLocationID
        ORDER BY trip_count DESC
    LIMIT 20
    ) TO "../data/marts/mart_duck_dropoff.parquet"
    """
)

In [137]:
#Distance distribution
diss.execute (
    """ 
    COPY (
        WITH binned_data AS (
        SELECT
            passenger_count,
            trip_distance,
            total_amount,
            tip_rate,
            duration_mins,
            
            CASE 
                WHEN trip_distance >= 0 AND trip_distance < 1 THEN '0-1 mile'
                WHEN trip_distance >= 1 AND trip_distance < 3 THEN '1-3 miles'
                WHEN trip_distance >= 3 AND trip_distance < 7 THEN '3-7 miles'
                WHEN trip_distance >= 7 AND trip_distance < 15 THEN '5-10 miles'
                WHEN trip_distance >= 15 THEN '15+ miles'
            END AS distance_group
        FROM clean_taxinyc)
        
        SELECT
            distance_group,
            SUM(passenger_count) AS total_passengers,
            COUNT(*) AS trip_count,
            AVG(trip_distance) AS avg_trip_distance,
            SUM(total_amount) AS revenue,
            AVG(tip_rate) * 100 AS avg_tip_rate,
            AVG(duration_mins) AS avg_duration
        FROM binned_data
        GROUP BY distance_group
        ORDER BY trip_count DESC
    ) TO "../data/marts/mart_duck_dist.parquet"
    """
)

In [ ]:
#Basic
diss.execute (
    """ 
    COPY (
        
    ) TO "../data/marts/mart_duck_daily.parquet"
    """
)

In [5]:
diss.close()